# 🎬 영화 상세 정보 수집 (insert_movie.ipynb)

## 실행 전 반드시 확인

| 항목 | 내용 |
|------|------|
| **선행 조건** | `insert_box_office.ipynb`를 먼저 실행하여 `daily_box_office` 테이블에 데이터가 있어야 합니다 |
| **실행 방법** | 상단 메뉴 → `Run All` (셀을 순서대로 전부 실행) |
| **중단 시 재실행** | 중간에 끊겨도 **그냥 Run All** 하면 됩니다. 이미 저장된 영화는 자동으로 건너뜁니다 |
| **소요 시간** | 영화 2,000편 기준 약 20~30분 (API 호출 지연 포함) |

---

## 이 노트북이 하는 일

```
daily_box_office 테이블
        ↓  (movies에 없는 movie_id만 추출)
KOBIS API 호출 (영화 1편당 1회)
        ↓
데이터 가공
        ↓
movies 테이블         ← 영화 기본 정보 + 영화인 이름 JSON
companys 테이블       ← 제작사/배급사/제공사 정보
company_part 테이블   ← 영화-영화사 관계
```

---

## 수집 전략 요약

### 👤 영화인 (People) — 이름만 저장
KOBIS 영화 상세 API는 감독/배우의 고유 코드(`peopleCd`)를 제공하지 않습니다.  
→ 이름만 JSON으로 저장하고, 고유 ID 매핑은 `insert_people.ipynb`에서 별도 처리합니다.
```json
{"directors": ["봉준호"], "actors": ["송강호", "이선균"]}
```

### 🏢 영화사 (Company) — 즉시 저장
API가 고유 코드(`companyCd`)를 제공하므로 바로 저장합니다.  
→ **제작사, 배급사, 제공** 역할만 필터링하여 저장합니다.

### ⚠️ 결측값 처리 기준
API 응답이 없는 필드는 아래 기본값으로 채웁니다.

| 컬럼 | 기본값 | 이유 |
|------|--------|------|
| `genre` | `'미정'` | NOT NULL 컬럼, 문자열 타입 |
| `rating` | `'미정'` | NOT NULL 컬럼, 문자열 타입 |
| `nation` | `'미정'` | NOT NULL 컬럼, 문자열 타입 |
| `open_date` | `'1900-01-01'` | NOT NULL 컬럼, DATE 타입이라 문자열 불가 |
| `runtime` | `0` | NOT NULL 컬럼, INT 타입 |

---
## ⚙️ 셀 0. 라이브러리 임포트 및 기본 설정

필요한 라이브러리와 전역 상수를 정의합니다.

- `json` : 영화인/심의 정보를 JSON 문자열로 변환할 때 사용
- `time` : API 과호출 방지를 위한 딜레이 (`time.sleep`)
- `get_movie_info` : KOBIS 영화 상세 API 호출 함수
- `db` : 싱글톤 DBManager 인스턴스 (팀 공용 DB 객체)
- `CHUNK_SIZE` : 몇 편 단위로 중간 저장할지 결정. **변경 가능** (기본값: 50)
- `TARGET_PARTS` : 저장할 영화사 역할 필터. 이 3가지 외 역할은 저장하지 않음

In [ ]:
import json
import time

from data.api import get_movie_info
from data.db import db

# Checkpoint 단위 (영화 N편마다 중간 저장)
# 숫자를 줄이면 더 자주 저장 (안전), 늘리면 속도 향상
CHUNK_SIZE = 50

# 저장할 영화사 역할 필터 (이 3가지 외의 역할은 저장하지 않음)
TARGET_PARTS = ["제작사", "배급사", "제공"]

print("✅ 설정 완료")

---
## 📋 셀 1. 수집 대상 movie_id 추출

`daily_box_office`에는 있지만 `movies`에는 아직 없는 영화 ID만 가져옵니다.

```sql
-- LEFT JOIN 후 movies 쪽이 NULL인 행 = 아직 수집 안 된 영화
SELECT DISTINCT dbo.movie_id
FROM daily_box_office dbo
LEFT JOIN movies m ON dbo.movie_id = m.movie_id
WHERE m.movie_id IS NULL
```

> 💡 **중단 후 재실행해도 안전한 이유**: 이미 저장된 영화는 `movies`에 존재하므로  
> LEFT JOIN 조건에서 자동으로 제외됩니다. 항상 **남은 것만** 가져옵니다.

In [ ]:
query = """
    SELECT DISTINCT dbo.movie_id
    FROM daily_box_office dbo
    LEFT JOIN movies m ON dbo.movie_id = m.movie_id
    WHERE m.movie_id IS NULL
    ORDER BY dbo.movie_id
"""

rows = db.fetch_all(query)
target_movie_ids = [row["movie_id"] for row in rows]

print(f"📋 수집 대상 영화 수: {len(target_movie_ids)}편")
print(f"   예시: {target_movie_ids[:5]}")

# 0편이 나왔다면?
# → daily_box_office가 비어있거나, 이미 모든 영화가 수집된 상태입니다.
# → insert_box_office.ipynb를 먼저 실행했는지 확인하세요.

---
## 🛠️ 셀 2. 데이터 가공 헬퍼 함수 정의

`parse_movie_info()` 함수는 KOBIS API 응답 객체를 받아 DB에 저장할 수 있는 형태로 변환합니다.

### 반환값 구조
```python
{
    'movie_row'        : tuple  # movies 테이블에 삽입할 1개 행
    'company_rows'     : list   # companys 테이블에 삽입할 행 목록
    'company_part_rows': list   # company_part 테이블에 삽입할 행 목록
}
```

### 주요 가공 로직
| 필드 | 처리 방식 |
|------|-----------|
| `genre` / `rating` / `nation` | 리스트의 첫 번째 값만 사용, 빈 리스트면 `'미정'` |
| `open_date` | `'YYYYMMDD'` → `'YYYY-MM-DD'` 변환, 없으면 `'1900-01-01'` |
| `runtime` | 문자열 → 정수 변환, 없거나 변환 불가하면 `0` |
| `audits` | 심의 번호 + 등급 전체를 JSON 배열로 저장 |
| `people` | 감독/배우 **이름만** JSON으로 저장 (ID는 insert_people.ipynb에서 처리) |
| 영화사 | `TARGET_PARTS`에 해당하는 역할만 필터링하여 저장 |

In [ ]:
def parse_movie_info(movie_id: str, movie_info) -> dict:
    """
    KOBIS API의 movieInfo 객체를 받아 DB 저장용 딕셔너리로 가공합니다.

    Args:
        movie_id  (str): KOBIS 영화 고유 코드 (예: '20124079')
        movie_info    : get_movie_info() 응답의 .movieInfoResult.movieInfo 객체

    Returns:
        dict: 'movie_row', 'company_rows', 'company_part_rows' 키를 가진 딕셔너리
    """
    # --- 기본 정보 추출 ---
    title = movie_info.movieNm

    # 장르: 첫 번째 장르명만 사용 (없으면 미정)
    genre = movie_info.genres[0].genreNm if movie_info.genres else '미정'

    # 관람등급: 첫 번째 등급명만 사용 (없으면 미정)
    rating = movie_info.audits[0].watchGradeNm if movie_info.audits else '미정'

    # 제작국가: 첫 번째 국가명만 사용 (없으면 미정)
    nation = movie_info.nations[0].nationNm if movie_info.nations else '미정'

    # 개봉일: 'YYYYMMDD' → 'YYYY-MM-DD' 변환
    # DATE 타입 컬럼이라 문자열 '미정' 불가 → 1900-01-01을 미정 대체값으로 사용
    open_date_raw = movie_info.openDt
    if open_date_raw and len(open_date_raw) == 8:
        open_date = f"{open_date_raw[:4]}-{open_date_raw[4:6]}-{open_date_raw[6:8]}"
    else:
        open_date = '1900-01-01'

    # 런타임: 문자열 → 정수 변환 (변환 불가 또는 빈 값이면 0)
    runtime_raw = movie_info.showTm
    try:
        runtime = int(runtime_raw) if runtime_raw else 0
    except (ValueError, TypeError):
        runtime = 0

    # 심의 정보: auditNo(심의번호) + watchGradeNm(등급명) 전체를 JSON 배열로 저장
    # 예: [{"auditNo": "2019-...", "watchGradeNm": "15세이상관람가"}]
    audits_json = json.dumps(
        [{"auditNo": a.auditNo, "watchGradeNm": a.watchGradeNm} for a in movie_info.audits],
        ensure_ascii=False
    )

    # 영화인 정보: 이름만 JSON으로 저장
    # peopleCd(고유ID)는 이 단계에서 제공되지 않으므로 insert_people.ipynb에서 처리
    # 예: {"directors": ["봉준호"], "actors": ["송강호", "이선균"]}
    people_json = json.dumps(
        {
            "directors": [d.peopleNm for d in movie_info.directors],
            "actors": [a.peopleNm for a in movie_info.actors],
        },
        ensure_ascii=False
    )

    # movies 테이블 삽입용 tuple (아래 SQL 쿼리의 컬럼 순서와 반드시 일치해야 함)
    movie_row = (movie_id, title, genre, rating, nation, open_date, runtime, audits_json, people_json)

    # --- 영화사 정보 필터링 및 가공 ---
    company_rows = []       # companys 테이블용: (company_id, company_name)
    company_part_rows = []  # company_part 테이블용: (company_id, movie_id, part_role)

    for comp in movie_info.companys:
        # TARGET_PARTS에 해당하는 역할(제작사/배급사/제공)만 저장
        if comp.companyPartNm in TARGET_PARTS:
            company_rows.append((comp.companyCd, comp.companyNm))
            company_part_rows.append((comp.companyCd, movie_id, comp.companyPartNm))

    return {
        "movie_row": movie_row,
        "company_rows": company_rows,
        "company_part_rows": company_part_rows,
    }


print("✅ 헬퍼 함수 정의 완료")

---
## 💾 셀 3. Upsert SQL 쿼리 정의

3개 테이블에 대한 INSERT 쿼리를 상수로 정의합니다.

### `ON DUPLICATE KEY UPDATE`를 사용하는 이유
- **멱등성 보장**: 같은 데이터를 여러 번 실행해도 중복 삽입 없이 안전하게 업데이트됩니다.
- **재실행 안전**: 중단 후 재실행 시 이미 저장된 데이터는 덮어쓰기만 하고 오류가 나지 않습니다.

### 쿼리 파라미터 순서
| 쿼리 변수 | 파라미터 순서 |
|-----------|---------------|
| `SQL_INSERT_MOVIE` | `(movie_id, title, genre, rating, nation, open_date, runtime, audits, people)` |
| `SQL_INSERT_COMPANY` | `(company_id, company_name)` |
| `SQL_INSERT_COMPANY_PART` | `(company_id, movie_id, part_role)` |

In [ ]:
SQL_INSERT_MOVIE = """
    INSERT INTO movies (movie_id, title, genre, rating, nation, open_date, runtime, audits, people)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        title      = VALUES(title),
        genre      = VALUES(genre),
        rating     = VALUES(rating),
        nation     = VALUES(nation),
        open_date  = VALUES(open_date),
        runtime    = VALUES(runtime),
        audits     = VALUES(audits),
        people     = VALUES(people)
"""

SQL_INSERT_COMPANY = """
    INSERT INTO companys (company_id, company_name)
    VALUES (%s, %s)
    ON DUPLICATE KEY UPDATE
        company_name = VALUES(company_name)
"""

SQL_INSERT_COMPANY_PART = """
    INSERT INTO company_part (company_id, movie_id, part_role)
    VALUES (%s, %s, %s)
    ON DUPLICATE KEY UPDATE
        part_role = VALUES(part_role)
"""

print("✅ SQL 쿼리 정의 완료")

---
## 🚀 셀 4. 메인 수집 루프 (Checkpoint 방식)

핵심 실행 셀입니다. `CHUNK_SIZE`(기본 50)편씩 묶어서 API를 호출하고, 한 묶음이 끝날 때마다 DB에 저장합니다.

### 처리 흐름
```
for 50편씩:
    for 영화 1편씩:
        API 호출 → 가공 → 버퍼에 추가
        실패 시: failed_ids에 기록하고 다음 영화로 넘어감 (전체 중단 안 함)

    버퍼 → DB Bulk Insert (Checkpoint)
    DB 저장 실패 시: 치명적 오류이므로 전체 중단
```

### 주요 변수
| 변수 | 설명 |
|------|------|
| `buf_movie_rows` | 청크 내 movies 삽입 버퍼 |
| `buf_company_rows` | 청크 내 companys 삽입 버퍼 |
| `buf_company_part_rows` | 청크 내 company_part 삽입 버퍼 |
| `unique_companies` | company_id 기준 중복 제거된 영화사 목록 |
| `failed_ids` | API 호출 실패한 movie_id 목록 → 셀 5에서 재시도 |
| `time.sleep(0.05)` | API 서버 과부하 방지용 딜레이 (초 단위, 필요 시 조정) |

In [ ]:
total = len(target_movie_ids)
success_count = 0
fail_count = 0
failed_ids = []  # API 호출 실패한 movie_id 목록 (셀 5에서 재시도)

# CHUNK_SIZE 단위로 분할하여 처리
for chunk_start in range(0, total, CHUNK_SIZE):
    chunk_ids = target_movie_ids[chunk_start : chunk_start + CHUNK_SIZE]
    chunk_end = min(chunk_start + CHUNK_SIZE, total)

    # 청크 내 버퍼 초기화 (청크마다 새로 시작)
    buf_movie_rows        = []
    buf_company_rows      = []
    buf_company_part_rows = []

    print(f"\n📦 [{chunk_start + 1} ~ {chunk_end} / {total}] API 호출 시작...")

    for movie_id in chunk_ids:
        try:
            # KOBIS API 호출: movie_cd로 영화 상세 정보 조회
            response = get_movie_info(movie_cd=movie_id)
            movie_info = response.movieInfoResult.movieInfo

            # API 응답을 DB 저장 형태로 가공
            parsed = parse_movie_info(movie_id, movie_info)

            buf_movie_rows.append(parsed["movie_row"])
            buf_company_rows.extend(parsed["company_rows"])
            buf_company_part_rows.extend(parsed["company_part_rows"])

            success_count += 1

        except Exception as e:
            # 개별 영화 실패는 기록만 하고 다음 영화로 계속 진행
            print(f"  ⚠️  movie_id={movie_id} 실패: {e}")
            failed_ids.append(movie_id)
            fail_count += 1
            continue

        # API 과호출 방지 딜레이 (0.05초 = 50ms)
        time.sleep(0.05)

    # ── Checkpoint: 청크 단위 Bulk Insert ──
    # execute_many()는 내부적으로 1000건 단위로 나눠서 처리함 (db_manager.py 참고)
    try:
        if buf_movie_rows:
            db.execute_many(SQL_INSERT_MOVIE, buf_movie_rows)

        # 같은 company_id가 여러 영화에 등장할 수 있으므로 중복 제거 후 삽입
        # {company_id: (company_id, company_name)} 딕셔너리로 중복 제거
        unique_companies = list({row[0]: row for row in buf_company_rows}.values())
        if unique_companies:
            db.execute_many(SQL_INSERT_COMPANY, unique_companies)

        if buf_company_part_rows:
            db.execute_many(SQL_INSERT_COMPANY_PART, buf_company_part_rows)

        print(f"  ✅ Checkpoint 저장 완료 — movies: {len(buf_movie_rows)}편, "
              f"companys: {len(unique_companies)}개, "
              f"company_part: {len(buf_company_part_rows)}건")

    except Exception as e:
        # DB 저장 실패는 데이터 유실 위험이 있으므로 전체 중단
        print(f"  ❌ Checkpoint DB 저장 실패: {e}")
        raise

print(f"\n{'='*50}")
print(f"🎉 수집 완료!")
print(f"   성공: {success_count}편 / 실패: {fail_count}편")
if failed_ids:
    print(f"   실패 목록: {failed_ids}")
    print(f"   → 아래 셀 5를 실행하여 재시도하세요.")

---
## 🔁 셀 5. 실패 재시도 (선택 실행)

> ⚠️ **셀 4에서 실패한 영화가 있을 때만 실행하세요.**  
> `failed_ids`가 비어 있으면 자동으로 "실패 없음" 메시지를 출력하고 종료됩니다.

재시도 시 딜레이를 0.1초로 늘려 네트워크 부하를 줄입니다.  
재시도 후에도 실패한 `movie_id`는 **최종 실패 목록**으로 출력됩니다.

In [ ]:
if not failed_ids:
    print("✅ 실패한 영화가 없습니다.")
else:
    print(f"🔁 {len(failed_ids)}편 재시도 중...")
    retry_success = []
    retry_fail    = []

    retry_movie_rows        = []
    retry_company_rows      = []
    retry_company_part_rows = []

    for movie_id in failed_ids:
        try:
            response   = get_movie_info(movie_cd=movie_id)
            movie_info = response.movieInfoResult.movieInfo
            parsed     = parse_movie_info(movie_id, movie_info)

            retry_movie_rows.append(parsed["movie_row"])
            retry_company_rows.extend(parsed["company_rows"])
            retry_company_part_rows.extend(parsed["company_part_rows"])
            retry_success.append(movie_id)

        except Exception as e:
            print(f"  ⚠️  재시도 실패 movie_id={movie_id}: {e}")
            retry_fail.append(movie_id)

        # 재시도는 딜레이를 늘려서 실행 (네트워크 안정성 확보)
        time.sleep(0.1)

    # 재시도 성공분 DB 저장
    if retry_movie_rows:
        db.execute_many(SQL_INSERT_MOVIE, retry_movie_rows)

    unique_retry_companies = list({row[0]: row for row in retry_company_rows}.values())
    if unique_retry_companies:
        db.execute_many(SQL_INSERT_COMPANY, unique_retry_companies)

    if retry_company_part_rows:
        db.execute_many(SQL_INSERT_COMPANY_PART, retry_company_part_rows)

    print(f"\n재시도 결과 — 성공: {len(retry_success)}편 / 최종 실패: {len(retry_fail)}편")
    if retry_fail:
        print(f"최종 실패 목록: {retry_fail}")
        print("→ 위 movie_id들은 KOBIS에 정보가 없는 영화일 수 있습니다. 수동으로 확인해 보세요.")

---
## ✅ 셀 6. 수집 결과 검증

수집이 끝난 후 DB에 데이터가 제대로 들어갔는지 확인합니다.

### 정상 결과 기준
- **미수집 영화 0편** → 완전히 수집 완료
- **미수집 영화 N편** → 해당 movie_id는 KOBIS에 정보가 없는 영화일 가능성이 높음

In [ ]:
# 각 테이블 전체 건수 확인
movie_cnt = db.fetch_one("SELECT COUNT(*) AS cnt FROM movies")["cnt"]
comp_cnt  = db.fetch_one("SELECT COUNT(*) AS cnt FROM companys")["cnt"]
part_cnt  = db.fetch_one("SELECT COUNT(*) AS cnt FROM company_part")["cnt"]

print(f"📊 movies 테이블      : {movie_cnt}건")
print(f"📊 companys 테이블    : {comp_cnt}건")
print(f"📊 company_part 테이블: {part_cnt}건")

# 아직 movies에 없는 movie_id 확인 (0이면 완전 수집)
remaining = db.fetch_all("""
    SELECT DISTINCT dbo.movie_id
    FROM daily_box_office dbo
    LEFT JOIN movies m ON dbo.movie_id = m.movie_id
    WHERE m.movie_id IS NULL
""")
print(f"\n⚠️  미수집 영화: {len(remaining)}편")
if remaining:
    print("  movie_ids:", [r["movie_id"] for r in remaining])

In [ ]:
# 최근 수집된 영화 샘플 확인 (open_date 내림차순 10건)
# open_date가 1900-01-01인 항목은 개봉일 정보가 없는 영화입니다
sample = db.fetch_all("""
    SELECT movie_id, title, genre, rating, nation, open_date, runtime
    FROM movies
    ORDER BY open_date DESC
    LIMIT 10
""")

print("최근 수집된 영화 샘플 (최대 10건):")
for row in sample:
    print(f"  [{row['movie_id']}] {row['title']} | {row['genre']} | {row['rating']} | {row['open_date']} | {row['runtime']}분")